In [1]:
#Notebook formatting
from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -50% !important; margin-right: -50% !important; }</style>"))

### Project Process Note - Initial Setup & Data Wrangling Phase

This notebook represents the initial setup and data wrangling phase of the Lending Club credit risk project.

The initial data sets are large and arrived with inconsistent data types, mixed formatting, and many sparse or noisy columns. I loaded the data with Dask, using all columns initially as strings to avoid automatic inference errors and maintain full control over cleaning. 

This required exposure to Dask and learning how to handle inference issues common with large, real-world datasets.

Key work completed in this phase:
- Iteratively diagnosed and resolved repeated dtype mismatch errors across dozens of columns.
- Converted columns (`dti`, `emp_length`, other financial metrics) from string to appropriate numeric types.
- Dropped clearly low-value columns (e.g., `url`, `id`, `desc`, `emp_title`, `zip_code`, `policy_code`, etc.) while preserving columns useful for exploratory analysis. I kept sparse but potentially insightful columns like the hardship and settlement fields.
- Created manageable random samples for testing and validation **Lending Club Workbook.xlsx**
- Adjusted the approach several times to keep the process stable for memory.

The cleaning process took several attempts — I revisited and refined steps as new issues surfaced. This notebook focuses on getting the data into a usable state. 

After wrangling the data into a usable state, I will create additional notebooks for the subsequent project stages, including exploratory analysis and insights, modeling and prediction, and any further extensions as needed.

I used **Grok** as an AI assistant for code suggestions and troubleshooting, similar to collaborating with a senior colleague. However, all strategic decisions regarding what to clean, what to keep, column prioritization, and overall project structure were made by me.

In [ ]:
#Setup notes:
#Given that this project uses large data sets, I figured I need to look for extra resources not covered in the course content. So I'll be using Dask for the first time. 

#!pip install jupyter-resource-usage
#!jupyter server extension enable --py jupyter_resource_usage --sys-prefix
#!pip install memory_profiler --quiet
#%load_ext memory_profiler
#%unload_ext memory_profiler

#!pip install ipython-autotime --quiet
#%load_ext autotime
#%unload_ext autotime

In [ ]:
%load_ext memory_profiler

In [ ]:
%load_ext autotime

In [ ]:
#Function for memory profiling
def memit_gb():
    from memory_profiler import memory_usage
    mem = memory_usage()[0] / 1024   # convert MiB to GB
    print(f"Usage: {mem:.2f} GB")

memit_gb()

In [ ]:

import sys
import platform
import numpy as np
import pandas as pd
import time
import warnings
import dask.dataframe as dd
import gc
import seaborn as sns
sns.set_style("whitegrid")
# Noting libraries to import later:
#import matplotlib.pyplot as plt



#from sklearn.model_selection import train_test_split
#from sklearn.ensemble import RandomForestClassifier
#from sklearn.linear_model import LogisticRegression
#from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
#from sklearn.preprocessing import StandardScaler, OneHotEncoder
#from sklearn.compose import ColumnTransformer
#from sklearn.pipeline import Pipeline

# Utilities
import warnings
warnings.filterwarnings('ignore')

print('imported')

In [ ]:
#Loaded the raw data with all columns as string (dtype='object') to avoid inference errors, then manually converted key columns to appropriate numeric, further down.
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

dfa = dd.read_csv('LCA_2007-2018.csv', dtype='object')
dfr = dd.read_csv('LCR_2007-2018.csv', dtype='object')

In [ ]:
# ================================================
# DATA CLEANING / WRANGLING
# Starting to clean and standardize the data
# (renaming columns, fixing data types, handling missing values, etc.)
# ================================================

In [ ]:
dfa.head()

In [ ]:
dfr.head()

In [ ]:
%%memit
dfr = dfr.rename(columns={
    "Amount Requested": "amount_requested",
    "Application Date": "application_date",
    "Loan Title": "loan_title",
    "Risk_Score": "risk_score",
    "Debt-To-Income Ratio": "dti",
    "Zip Code": "zip_code",
    "State": "state",
    "Employment Length": "emp_length",
    "Policy Code": "policy_code"
})
#Rename Rejected loan columns. Convertring "Debt-To-Income Ratio" to dti for name match in both, the dfa and dfr tables. 

In [ ]:
print("Converting DFA columns")

# Float32 columns (percentages, ratios, rates, scores)
float32s = ['dti', 'il_util', 'bc_util', 'pct_tl_nvr_dlq', 
            'percent_bc_gt_75', 'settlement_percentage', 'revol_util',
            'dti_joint', 'int_rate', 'last_fico_range_high', 
            'last_fico_range_low', 'sec_app_fico_range_low', 
            'sec_app_fico_range_high', 'all_util']

# Currency-style columns (rounded to 2 decimals)
currency_cols = ['loan_amnt', 'installment', 'annual_inc', 'revol_bal',
                 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
                 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
                 'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt',
                 'tot_coll_amt', 'tot_cur_bal', 'total_bal_il', 'max_bal_bc',
                 'total_rev_hi_lim', 'avg_cur_bal', 'total_bal_ex_mort',
                 'total_bc_limit', 'total_il_high_credit_limit', 'revol_bal_joint',
                 'hardship_amount', 'hardship_payoff_balance_amount',
                 'hardship_last_payment_amount', 'settlement_amount',
                 'orig_projected_additional_accrued_interest',
                 'bc_open_to_buy', 'tot_hi_cred_lim']

# Integer columns
int_cols = ['delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'total_acc',
            'collections_12_mths_ex_med', 'acc_now_delinq', 'open_acc_6m',
            'open_act_il', 'open_il_12m', 'open_il_24m', 'open_rv_12m', 
            'open_rv_24m', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
            'acc_open_past_24mths', 'chargeoff_within_12_mths', 'mort_acc',
            'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
            'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
            'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats',
            'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m',
            'num_tl_op_past_12m', 'pub_rec_bankruptcies', 'tax_liens',
            'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc',
            'sec_app_open_act_il', 'sec_app_num_rev_accts',
            'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med',
            'deferral_term', 'hardship_length', 'hardship_dpd', 'settlement_term']

# Date columns
dates_bom = [
    'issue_d', 'earliest_cr_line', 'last_credit_pull_d',
    'sec_app_earliest_cr_line', 'hardship_start_date',
    'payment_plan_start_date', 'debt_settlement_flag_date']

dates_eom = ['last_pymnt_d', 'next_pymnt_d', 'hardship_end_date', 'settlement_date']


for col in float32s:
    if col in dfa.columns:
        dfa[col] = dfa[col].map_partitions(lambda x: pd.to_numeric(x, errors='coerce'), meta=(col, 'float32'))

for col in currency_cols:
    if col in dfa.columns:
        dfa[col] = dfa[col].map_partitions(lambda x: pd.to_numeric(x, errors='coerce').round(2), meta=(col, 'float32'))

for col in int_cols:
    if col in dfa.columns:
        dfa[col] = dfa[col].map_partitions(lambda x: pd.to_numeric(x, errors='coerce').astype('Int32'), meta=(col, 'Int32'))

for col in dates_bom:
    if col in dfa.columns:
        dfa[col] = dfa[col].map_partitions(
            lambda x: pd.to_datetime(x, errors='coerce').dt.to_period('M').dt.to_timestamp(), 
            meta=(col, 'datetime64[ns]')
        )

for col in dates_eom:
    if col in dfa.columns:
        dfa[col] = dfa[col].map_partitions(
            lambda x: pd.to_datetime(x, errors='coerce').dt.to_period('M').dt.to_timestamp('M'), 
            meta=(col, 'datetime64[ns]')
        )

print("Converted")

In [ ]:
#Checking .head() values to ensure type conversions look correct. 
dfa.head()

In [ ]:
#Deleting some columns that are mostly redundant, useless, or contain similar sub-categories with more granular, useful data in other columns.
dfa = dfa.drop(columns=[
    'url', 
    'member_id', 
    'id', 
    'desc', 
    'title', 
    'emp_title', 
    'zip_code', 
    'policy_code',
    'pymnt_plan',
    'initial_list_status',
    'disbursement_method',
    'funded_amnt',
    'funded_amnt_inv',
    'grade'
    
])

In [ ]:
#Convert DTI string to numeric float

dfr['dti'] = dfr['dti'].astype(str).str.replace('%', '').str.strip()
dfr['dti'] = dfr['dti'].map_partitions(lambda x: pd.to_numeric(x, errors='coerce') / 100, meta=('dti', 'float64'))

print("dti complete")
print("\n dfa [dti] head:")
print(dfa['dti'].head(10))
print("\n dfr [dti] head:")
print(dfr['dti'].head(10))

In [ ]:
#Check data types
display(dfr.dtypes, dfa.dtypes)

In [ ]:
#Cleaning/converting a few more columns.

dfa['emp_length'] = dfa['emp_length'].astype(str)
dfa['emp_length'] = dfa['emp_length'].str.extract(r'(\d+)')[0].astype(int)
dfa['emp_length'] = dfa['emp_length'].fillna(0)

dfr['emp_length'] = dfr['emp_length'].astype(str)
dfr['emp_length'] = dfr['emp_length'].str.extract(r'(\d+)')[0].astype(int)
dfr['emp_length'] = dfr['emp_length'].fillna(0)

In [ ]:
dfa['emp_length'].head(10)

In [ ]:
#Memory check
import gc
gc.collect()

print("Current memory usage after cleanup:")
memit_gb()

print("\nCurrent number of columns in dfa:", len(dfa.columns))
print("Current number of columns in dfr:", len(dfr.columns))

In [ ]:
dfa.head()

In [ ]:
#Memory check:
import gc
import gc
gc.collect()
print("MEMORY: ")
memit_gb() 

In [ ]:
print("Cleaning term column:")

if 'term' in dfa.columns:
    dfa['term'] = dfa['term'].astype(str).str.extract(r'(\d+)')[0].astype('Int32')
    print("term cleaned on dfa")
else:
    print("term column not found in dfa")

print("Complete")

In [ ]:
#Memory check:
import gc
gc.collect()
print("MEMORY: ")
memit_gb() 

In [ ]:
#Fill NaN / <N/A> columns with 0.00 in currency columns
for col in currency_cols:
    if col in dfa.columns:
        dfa[col] = dfa[col].fillna(0)
        dfa[col] = dfa[col].round(2)
print('complete')

In [ ]:
#Fill float columns NaNs / <NA> with 0.0
for col in float32s:
    if col in dfa.columns:
        dfa[col] = dfa[col].fillna(0)
print('complete')

In [ ]:
#Fill integer columns NaNs / <NA> with 0
for col in int_cols:
    if col in dfa.columns:
        dfa[col] = dfa[col].fillna(0) 
print('complete')

In [ ]:
dfa.head(10)

### To-Do List - Data Wrangling Phase - 5/6/2026

- [x] Fill remaining NaN / `<NA>` values with 0 or 0.0 across numeric columns
- [x] Convert strings to appropriate numeric and datetime types
- [x] Decide BOM vs EOM logic for date columns
- [x] Force currency-style columns to show two decimal places
- [ . ] Final memory optimization pass before export
- [ . ] Export cleaned dataset as Parquet/CSV for next notebook
- [ . ] Clean DFR columns

In [ ]:
# ===================================================================
# UTILITIES / SCRATCHPAD
# Random one-liners and helper code I might want later.
# Keep the main notebook clean.
# ===================================================================

# Quick memory cleanup
import gc
gc.collect()

# Delete large objects when done with them
# del dfa, dfr, dfa_sample, dfr_sample

# Example exports (uncomment when needed)
# dfa_sample.to_excel('LCA_10k.xlsx', index=False)
# dfr_sample.to_excel('LCR_10k.xlsx', index=False)

# Quick inspection examples
# dfa.head(20)
# dfa['loan_status'].value_counts().compute()
# dfr.head(10)
# ===================================================================
# MEMORY TEST SNIPPETS - dfa and dfr - column cleaning tests
# [using dti column for test example]
# ===================================================================

# Test 0.1% ----------------------------------------------------------------
testdfa1 = dfa.sample(frac=0.001, random_state=42).compute()
testdfa1['dti'] = testdfa1['dti'].str.replace('%', '').astype(float).round(2)
print("0.1\'%' - dfa - ")
del testdfa1
gc.collect()
# Test 2% ----------------------------------------------------------------
testdfa2 = dfa.sample(frac=0.02, random_state=42).compute()
testdfa2['dti'] = testdfa2['dti'].str.replace('%', '').astype(float).round(2)
print("2\'%' - dfa - ")
del testdfa2
gc.collect()
# Test 10% ----------------------------------------------------------------
testdfa3 = dfa.sample(frac=0.1, random_state=42).compute()
testdfa3['dti'] = testdfa3['dti'].str.replace('%', '').astype(float).round(2)
print("10\'%' - dfa - ")
del testdfa3
gc.collect()
# Test 20% ----------------------------------------------------------------
testdfa4 = dfa.sample(frac=0.2, random_state=42).compute()
testdfa4['dti'] = testdfa4['dti'].str.replace('%', '').astype(float).round(2)
print("20\'%' - dfa - ")
del testdfa4
gc.collect()
# Test 50% ----------------------------------------------------------------
testdfa5 = dfa.sample(frac=0.5, random_state=42).compute()
testdfa5['dti'] = testdfa5['dti'].str.replace('%', '').astype(float).round(2)
print("50\'%' - dfa - ")
del testdfa5
gc.collect()

# Junk for later:
head_df = dfa.head(10)
display(head_df)
print(f"Time taken: {clock} seconds")



